# Fase 2 — Otimização de rotas médicas (Projeto 2)

Executa localmente a otimização nos cenários `demo`, `olist_sp` e `hybrid_sp`, compara baseline e plano otimizado, persiste histórico e gera relatório local. CNES pode ser escolhido na interface se houver CSV autorizado instalado. As células batch não precisam de API ou chave. A interface ao final permite visualizar mapa, manifesto e consultar Gemini quando a API estiver configurada.

Para a interface opcional ao final, inicie o dashboard/API local na raiz do repositório com `.\run.ps1` e mantenha o terminal aberto. A interface do notebook envia requisições somente para `http://127.0.0.1:8020`; ela não solicita nem guarda chaves de provedor.

**Limites e privacidade:** no cenário Olist, municípios e pedidos históricos são agregados de comércio eletrônico. Frota, carga médica, prioridades e prazos são simulados; no demo os pontos também são fictícios. Distâncias são aproximações em linha reta, não rotas viárias. Não informe dados de pacientes ou identificadores pessoais.

In [ ]:
from pathlib import Path
import sys

candidates = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (path for path in candidates if (path / "src" / "optimization" / "notebook_ui.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Abra o notebook dentro do repositório do projeto.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Projeto: {PROJECT_ROOT}")

In [ ]:
import pandas as pd
from IPython.display import display
from src.logistics.genetic import GAConfig
from src.logistics.service import run_route_optimization, load_route_plan, route_history_summary

plans = {}
rows = []
for source in ('demo', 'olist_sp', 'hybrid_sp'):
    plan = run_route_optimization(GAConfig(population_size=16, generations=20, seed=42), source=source)
    plans[source] = plan
    optimized = plan['comparison']['optimized']
    assert not optimized['unserved_delivery_ids']
    assert load_route_plan(source)['scenario_id'] == plan['scenario_id']
    rows.append({'source': source, 'baseline_km': plan['comparison']['baseline']['total_distance_km'], 'optimized_km': optimized['total_distance_km'], 'saved_percent': plan['comparison']['distance_saved_percent']})
display(pd.DataFrame(rows))
print('Configuração demonstrativa reduzida; não garante ótimo global ou economia real.')


In [ ]:
import os
from unittest.mock import patch
from src.logistics.llm import generate_route_content

history = route_history_summary(source='hybrid_sp')
assert history['run_count'] >= 1
with patch.dict(os.environ, {}, clear=True):
    report = generate_route_content(plans['hybrid_sp'], action='daily_report')
assert report['source'] == 'local'
display(report['content'])
print('Relatório por regras, sem Gemini. Histórico de simulações, não entregas realizadas:', history['run_count'])


## Carregar cenário e resultados

A fonte Olist fica selecionada inicialmente. Se os CSVs não estiverem disponíveis ou válidos, o notebook mostrará o erro da API e não trocará silenciosamente para dados fictícios. Escolha `demo` para testar sem os arquivos Olist. Se já houver uma otimização salva para a fonte selecionada, ela será exibida; caso contrário, ajuste os parâmetros e execute o algoritmo pelo botão.

In [ ]:
from src.optimization.notebook_ui import display_route_notebook

route_ui = display_route_notebook()